# Model-Agnostic Task Complexity Estimator
### Method 1: Structural Feature Score — No LLM Inference Required

---

**Framework:** Composite weighted score over 5 orthogonal feature dimensions extracted from raw task text using classical NLP.

$$C(q) = w_S \cdot S + w_R \cdot R + w_T \cdot T + w_D \cdot D + w_{TT} \cdot TT \quad \in [0, 1]$$

| Layer | Dimension | Description |
|-------|-----------|-------------|
| L1 | **S** — Surface Features | Token count, lexical diversity (MATTR), NER density |
| L2 | **R** — Reasoning Depth | Bloom level, syntactic tree depth, multi-hop, negation, conditionals |
| L3 | **T** — Tool Dependency | Number of distinct tool/API calls needed |
| L4 | **D** — Domain Ski`lls | Domain breadth, temporal reasoning |
| L5 | **TT** — Task Type | Generative > Reasoning > Alt-choice > Single > Decision |

**Key principle:** Zero LLM calls. Every feature is derived from deterministic NLP signals.

## Cell 0 — Install Dependencies

1. sys.executable ensures it uses the same Python
2. check_call → throws an error if installation fails
3. Downloads en_core_web_md, a medium-sized English NLP model --> pretrained english language model --> used by spacy libray
   1. Includes:
      1. Tokenization : it breaks text into words/ sentences
      2. POS tagging : identified grammar roles
      3. Named Entity Recognition (NER) : find real world things
      4. Word vectors (semantic meaning) : it gives words numerical meaning representaion ; 'king' == 'queen'
   
   

In [1]:
# Run once — restart kernel after this cell completes
import subprocess, sys

packages = ["spacy", "textstat", "scipy", "numpy", "pandas", "matplotlib", "seaborn"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# Download spaCy model with parser + NER
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_md", "--quiet"])
print("✅ All dependencies installed.")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
✅ All dependencies installed.


## Cell 1 — Global Imports & spaCy Pipeline

In [2]:
from __future__ import annotations # used to allow type hints to be treated as strings

import re  # regex ( pattern mataching in text)
import math # math function 
import warnings ## controls warnings
from dataclasses import dataclass, field, asdict # create a objective structure 
from typing import Dict, List, Optional, Tuple # type hints --> for safe and cleaner code

import numpy as np # numerical operations
import pandas as pd # tables/dataframes
import matplotlib.pyplot as plt # plots
import matplotlib.patches as mpatches 
import seaborn as sns
import spacy # advanced NLP ( NER, parsing, vectors)
import textstat # readabilty scores

from pathlib import Path


warnings.filterwarnings("ignore")

# ── Load spaCy with ALL components needed across all layers ─────────────────
# We load once here at module level; downstream classes reuse this singleton.
_NLP = spacy.load("en_core_web_md")   # includes: tagger, parser, NER, vectors

print(f"✅ spaCy pipeline: {_NLP.pipe_names}")
print(f"✅ Model: en_core_web_md  |  Vocab: {len(_NLP.vocab):,} tokens")

✅ spaCy pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
✅ Model: en_core_web_md  |  Vocab: 764 tokens


## Cell 2 — Helper Utilities

In [3]:
def fix(value: float, lo: float = 0.0, hi: float = 1.0) -> float:
    """fix a value to [lo, hi]."""
    return max(lo, min(hi, value))


def min_max_scale(value: float, lo: float, hi: float) -> float:
    """Map value from [lo, hi] → [0, 1]. ."""
    if hi == lo:
        return 0.0
    return fix((value - lo) / (hi - lo))


def _tree_depth(token: spacy.tokens.Token) -> int: ## root as main verd/ not always a verd--> how other words are relate to it # [Subject] + [Verb] + [Object/Modifier]
    """Recursively compute depth of a token in the dependency tree."""
    if list(token.children):
        return 1 + max(_tree_depth(child) for child in token.children)
    return 1


def get_doc(text: str) -> spacy.tokens.Doc:
    return _NLP(text)


print("✅ Utility functions defined.")

✅ Utility functions defined.


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0 — Imports & Config
# ─────────────────────────────────────────────────────────────────────────────



DATA_PATHS: Dict[str, Path] = {
    'gaia'     : Path('../datasets/processed/gaia.parquet'),
    'aime'     : Path('../datasets/processed/aime.parquet'),
    'mmlu_pro' : Path('../datasets/processed/mmlu_pro.parquet'),
    'musique'  : Path('../datasets/processed/musique.parquet'),
    'swe_bench' : Path('../datasets/processed/swe_bench.parquet'),
}

OUTPUT_DIR = Path('data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Imports OK')
print(f'   pandas  {pd.__version__}')
for name, path in DATA_PATHS.items():
    print(f'   {name:<10} → {path}')

✅ Imports OK
   pandas  3.0.1
   gaia       → ../datasets/processed/gaia.parquet
   aime       → ../datasets/processed/aime.parquet
   mmlu_pro   → ../datasets/processed/mmlu_pro.parquet
   musique    → ../datasets/processed/musique.parquet
   swe_bench  → ../datasets/processed/swe_bench.parquet


In [5]:
## Datasets 
datasets: Dict[str, pd.DataFrame] = {}
for name, path in DATA_PATHS.items():
    df = pd.read_parquet(path)
    datasets[name] = df
    print(f"\n{'─' * 60}")
    print(f"📂 Dataset : {name.upper()}")
    print(f"   Path    : {path}")
    print(f"   Shape   : {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"   Columns : {list(df.columns)}")
    print(f"   Nulls   : {df.isnull().sum().sum():,} total missing values")
    # print(f"\n{df.head()}\n")

print(f"\n✅ Loaded {len(datasets)} datasets successfully")
sample= datasets['gaia']["query"][0]


────────────────────────────────────────────────────────────
📂 Dataset : GAIA
   Path    : ../datasets/processed/gaia.parquet
   Shape   : 165 rows × 7 cols
   Columns : ['id', 'query', 'answer', 'level', 'annotator_steps', 'annotator_tools', 'file_name']
   Nulls   : 0 total missing values

────────────────────────────────────────────────────────────
📂 Dataset : AIME
   Path    : ../datasets/processed/aime.parquet
   Shape   : 30 rows × 5 cols
   Columns : ['id', 'query', 'answer', 'year', 'solution']
   Nulls   : 0 total missing values

────────────────────────────────────────────────────────────
📂 Dataset : MMLU_PRO
   Path    : ../datasets/processed/mmlu_pro.parquet
   Shape   : 12,032 rows × 9 cols
   Columns : ['id', 'query', 'answer', 'answer_index', 'options', 'category', 'cot_content', 'src', 'cot_length']
   Nulls   : 0 total missing values

────────────────────────────────────────────────────────────
📂 Dataset : MUSIQUE
   Path    : ../datasets/processed/musique.parquet
   

## 1: Surface Features (S)

**Features:**
1.  `token_count` → raw word count --> How long the text is
2.  `ner_density` → entity tokens / total tokens   --> How many named entities (people, places ....)
3.  `mattr` → Moving-Average Type-Token Ratio (window=25); length-independent lexical diversity --> How diverse the vocabulary is

**Output S = [0, 1]:** weighted combination, normalised against empirically reasonable query bounds.

In [6]:
class SurfaceFeatures:
    """
    L1 — Surface Features.

    Computes three length/richness signals from raw query text without any
    model inference:
      - token_count   : raw word count
      - ner_density   : fraction of tokens that are named entities
      - mattr         : Moving-Average Type-Token Ratio (window=25)

    These are combined into a single score S ∈ [0, 1].
    """

    # # Normalisation bounds (empirically set for typical LLM benchmark queries)
    # _TC_LO, _TC_HI   = 5,   150   # token count range
    # _NER_LO, _NER_HI = 0.0, 0.25  # NER density range
    # _MATTR_LO        = 0.2        # below this → very repetitive text
    # _MATTR_HI        = 1.0


    _TC_LO, _TC_HI   = 8,   167   # token count range
    _NER_LO, _NER_HI = 0.0, 0.2899  # NER density range
    _MATTR_LO        = 0.7265        # below this → very repetitive text
    _MATTR_HI        = 1.0



    # Sub-feature weights (must sum to 1.0)
    _W_TC    = 0.45
    _W_NER   = 0.30
    _W_MATTR = 0.25

    def __init__(self, text: str, doc: Optional[spacy.tokens.Doc] = None):
        self.text = text
        self.doc  = doc if doc is not None else get_doc(text)

    # ── Private sub-features ─────────────────────────────────────────────────

    def _token_count(self) -> int:
        """Count non-punctuation, non-whitespace tokens."""
        return sum(1 for t in self.doc if not t.is_punct and not t.is_space)

    def _ner_density(self) -> float:
        """Fraction of tokens that belong to a named entity span."""
        total = max(len(self.doc), 1)
        entity_token_count = sum(len(ent) for ent in self.doc.ents)
        return round(entity_token_count / total, 4)

    def _mattr(self, window: int = 25) -> float:
        """
        Moving-Average Type-Token Ratio.
        Slides a fixed-length window across all tokens, computes TTR at each
        position, then returns the mean.  Length-independent unlike raw TTR.
        Falls back to plain TTR when text is shorter than the window.
        """
        tokens = [t.lower_ for t in self.doc if not t.is_punct and not t.is_space]
        n = len(tokens)
        if n == 0:
            return 0.0
        if n <= window:
            return round(len(set(tokens)) / n, 4)
        ttr_scores = [
            len(set(tokens[i : i + window])) / window
            for i in range(n - window + 1)
        ]
        return round(sum(ttr_scores) / len(ttr_scores), 4)

    def raw_features(self) -> dict:
        return {
            "token_count": self._token_count(),
            "ner_density": self._ner_density(),
            "mattr":       self._mattr(),
        }

    def score(self) -> Tuple[float, dict]:
        """
        Returns (S, raw_features).
        S is the normalised surface complexity ∈ [0, 1].
        """
        raw = self.raw_features()

        s_tc    = min_max_scale(raw["token_count"], self._TC_LO,    self._TC_HI)
        s_ner   = min_max_scale(raw["ner_density"], self._NER_LO,   self._NER_HI)
        s_mattr = min_max_scale(raw["mattr"],       self._MATTR_LO, self._MATTR_HI)

        S = self._W_TC * s_tc + self._W_NER * s_ner + self._W_MATTR * s_mattr
        return fix(S), raw



In [7]:

# ── Quick sanity check ───────────────────────────────────────────────────────

_sf   = SurfaceFeatures(sample)
_S, _r = _sf.score()
print(f"Demo query : '{sample}'")
print(f"Raw        : {_r}")
print(f"S score    : {_S:.4f}")

Demo query : 'A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure with three axes, where each axis has a label word at both ends. Which of these words is used to describe a type of society in a Physics and Society article submitted to arXiv.org on August 11, 2016?'
Raw        : {'token_count': 55, 'ner_density': 0.1864, 'mattr': 0.9097}
S score    : 0.4934


In [8]:
import math
import numpy as np
import pandas as pd
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────

FEATURES   = ["token_count", "ner_density", "mattr"]
STRATEGIES = {"P2_P98": (2, 98), "P5_P95": (5, 95), "P10_P90": (10, 90)}

# Option 2 — toggle per-dataset bounds instead of global
USE_DATASET_BOUNDS = False      # ← flip to True to activate

# Option 3 — toggle log scale for token_count
USE_LOG_SCALE_TC   = True      # ← flip to True to activate

# ── Option 3 : log scaler (replaces min_max_scale for token_count) ────────────

def log_scale(value: float, lo: float, hi: float) -> float:
    """
    Compresses wide token-count ranges (e.g. 8–2417).
    Midpoint shifts down so short queries get more room to breathe.

    Linear:  token=8 → 0.0   token=87  → 0.5   token=167 → 1.0
    Log:     token=8 → 0.0   token=37  → 0.5   token=167 → 1.0
    """
    value = max(lo, min(hi, value))                         # clip to [lo, hi]
    return (math.log(value) - math.log(lo)) / (math.log(hi) - math.log(lo))

# ── Step 1 : Extract ──────────────────────────────────────────────────────────

def extract_features(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for name, df in datasets.items():
        print(f"⚙  {name.upper()} — {len(df):,} queries")
        for _, row in tqdm(df.iterrows(), total=len(df), leave=False):
            sf  = SurfaceFeatures(str(row["query"]))
            raw = sf.raw_features()
            rows.append({"dataset": name, **raw})
    return pd.DataFrame(rows)


feature_df = extract_features(datasets)
print(f"\n✅ {len(feature_df):,} queries extracted\n")

# ── Step 2 : Percentile report ──
## for each feature : meam, std, min/max. percentails  --> If 95% of token_count < 200 → good upper bound
def percentile_report(df: pd.DataFrame, label: str = "GLOBAL") -> pd.DataFrame:
    pcts = [1, 5, 10, 25, 50, 75, 90, 95, 99]
    rows = []
    for feat in FEATURES:
        s = df[feat]
        rows.append({
            "feature" : feat,
            "mean"    : s.mean().round(4),
            "std"     : s.std().round(4),
            "min"     : s.min().round(4),
            "max"     : s.max().round(4),
            **{f"p{p}": round(np.percentile(s, p), 4) for p in pcts},
        })
    report = pd.DataFrame(rows).set_index("feature")
    print(f"\n{'═'*60}\n  {label}\n{'═'*60}")
    print(report.to_string())
    return report


global_report = percentile_report(feature_df, "GLOBAL — all datasets pooled")

for name, group in feature_df.groupby("dataset"):
    percentile_report(group, f"{name.upper()} — {len(group):,} queries")

# ── Step 3 : Recommended bounds ─  

def recommended_bounds(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for feat in FEATURES:
        s   = df[feat]
        row = {"feature": feat}
        for label, (lo, hi) in STRATEGIES.items():
            row[f"{label}_LO"] = round(np.percentile(s, lo), 4)
            row[f"{label}_HI"] = round(np.percentile(s, hi), 4)
        rows.append(row)
    return pd.DataFrame(rows).set_index("feature")


# Global bounds (always computed — used when USE_DATASET_BOUNDS = False)
global_bounds = recommended_bounds(feature_df)
print(f"\n{'═'*60}\n  RECOMMENDED BOUNDS — GLOBAL\n{'═'*60}")
print(global_bounds.to_string())

# Option 2 — per-dataset bounds (computed always, applied only when flag is on)
dataset_bounds: dict[str, pd.DataFrame] = {}
for name, group in feature_df.groupby("dataset"):
    dataset_bounds[name] = recommended_bounds(group)
    print(f"\n{'═'*60}\n  RECOMMENDED BOUNDS — {name.upper()}\n{'═'*60}")
    print(dataset_bounds[name].to_string())

# ── Step 4 : Saturation audit ──

def saturation_audit(df: pd.DataFrame) -> pd.DataFrame: ## floor stauration --> too many values at maximum, Ceiling saturation → too many values at maximum
    current = {
        "token_count" : (SurfaceFeatures._TC_LO,    SurfaceFeatures._TC_HI),
        "ner_density" : (SurfaceFeatures._NER_LO,   SurfaceFeatures._NER_HI),
        "mattr"       : (SurfaceFeatures._MATTR_LO, SurfaceFeatures._MATTR_HI),
    }
    rows = []
    for feat, (lo, hi) in current.items():
        s = df[feat]
        rows.append({
            "feature"         : feat,
            "current_LO"      : lo,
            "current_HI"      : hi,
            "pct_floor (%)"   : round(100 * (s <= lo).mean(), 1),
            "pct_ceiling (%)" : round(100 * (s >= hi).mean(), 1),
        })
    audit = pd.DataFrame(rows).set_index("feature")
    print(f"\n{'═'*60}\n  SATURATION AUDIT  (>10% = adjust that bound)\n{'═'*60}")
    print(audit.to_string())
    return audit


audit = saturation_audit(feature_df)

# ── Step 5 : Apply bounds ─────────────────────────────────────────────────────

ATTR_MAP = {
    "token_count" : ("_TC_LO",    "_TC_HI"),
    "ner_density" : ("_NER_LO",   "_NER_HI"),
    "mattr"       : ("_MATTR_LO", "_MATTR_HI"),
}

def _write_bounds(bounds: pd.DataFrame, strategy: str, label: str = "") -> None:
    """Write one bounds table into SurfaceFeatures class attributes."""
    for feat, (lo_attr, hi_attr) in ATTR_MAP.items():
        setattr(SurfaceFeatures, lo_attr, bounds.loc[feat, f"{strategy}_LO"])
        setattr(SurfaceFeatures, hi_attr, bounds.loc[feat, f"{strategy}_HI"])
    if label:
        print(f"  → {label}  "
              f"TC=[{SurfaceFeatures._TC_LO}, {SurfaceFeatures._TC_HI}]  "
              f"NER=[{SurfaceFeatures._NER_LO}, {SurfaceFeatures._NER_HI}]  "
              f"MATTR=[{SurfaceFeatures._MATTR_LO}, {SurfaceFeatures._MATTR_HI}]")


def apply_bounds(strategy: str = "P5_P95") -> None:
    """
    Applies bounds and patches SurfaceFeatures.score() based on active options.

    Option 2 (USE_DATASET_BOUNDS): stores per-dataset bounds so score() picks
                                   the right ruler at inference time.
    Option 3 (USE_LOG_SCALE_TC):   patches score() to use log_scale instead of
                                   min_max_scale for token_count.
    """
    print(f"\nApplying {strategy} bounds "
          f"[dataset_bounds={USE_DATASET_BOUNDS}, log_scale_tc={USE_LOG_SCALE_TC}]")

    if USE_DATASET_BOUNDS:
        # Store per-dataset bounds as a class-level dict for lookup at score time
        SurfaceFeatures._dataset_bounds = {
            ds: {
                feat: (
                    float(bdf.loc[feat, f"{strategy}_LO"]),
                    float(bdf.loc[feat, f"{strategy}_HI"]),
                )
                for feat in FEATURES
            }
            for ds, bdf in dataset_bounds.items()
        }
        print("  Per-dataset bounds stored in SurfaceFeatures._dataset_bounds")
        for ds, bmap in SurfaceFeatures._dataset_bounds.items():
            print(f"    {ds:12s} : {bmap}")
    else:
        _write_bounds(global_bounds, strategy, label="global")

    # ── Option 3 : patch score() to use log_scale for token_count ────────────
    def patched_score(self, dataset: str = None) -> tuple[float, dict]:
        """
        Patched SurfaceFeatures.score().

        Args:
            dataset: required when USE_DATASET_BOUNDS=True.
                     Pass the dataset name so the right ruler is used.
                     Ignored when USE_DATASET_BOUNDS=False.
        """
        raw = self.raw_features()

        # ── Resolve bounds ────────────────────────────────────────────────────
        if USE_DATASET_BOUNDS and dataset is not None:
            bmap = self._dataset_bounds.get(dataset, None)
            if bmap is None:
                raise ValueError(
                    f"Unknown dataset '{dataset}'. "
                    f"Available: {list(self._dataset_bounds.keys())}"
                )
            tc_lo,  tc_hi  = bmap["token_count"]
            ner_lo, ner_hi = bmap["ner_density"]
            ma_lo,  ma_hi  = bmap["mattr"]
        else:
            tc_lo,  tc_hi  = self._TC_LO,    self._TC_HI
            ner_lo, ner_hi = self._NER_LO,   self._NER_HI
            ma_lo,  ma_hi  = self._MATTR_LO, self._MATTR_HI

        # ── Scale features ────────────────────────────────────────────────────
        s_tc    = (log_scale       if USE_LOG_SCALE_TC else min_max_scale)(
                      raw["token_count"], tc_lo, tc_hi)
        s_ner   = min_max_scale(raw["ner_density"], ner_lo, ner_hi)
        s_mattr = min_max_scale(raw["mattr"],       ma_lo,  ma_hi)

        S = self._W_TC * s_tc + self._W_NER * s_ner + self._W_MATTR * s_mattr
        return fix(S), raw

    # Bind patched method onto the class
    SurfaceFeatures.score = patched_score
    print("✅ Done.\n")


apply_bounds("P5_P95")


# ── Usage examples ────────────────────────────────────────────────────────────

text = "Find and fix the authentication bug in the login service."

# Default (global bounds, linear scale)
sf = SurfaceFeatures(text)
S, raw = sf.score()
print(f"Global / linear  → S={S:.4f}  raw={raw}")

# Option 2 — pass dataset name so the right ruler is used
if USE_DATASET_BOUNDS:
    S2, raw2 = sf.score(dataset="swe_bench")
    print(f"SWE_BENCH ruler  → S={S2:.4f}  raw={raw2}")

    S3, raw3 = sf.score(dataset="musique")
    print(f"MUSIQUE ruler    → S={S3:.4f}  raw={raw3}")

⚙  GAIA — 165 queries


⚙  AIME — 30 queries


⚙  MMLU_PRO — 12,032 queries


⚙  MUSIQUE — 2,417 queries


⚙  SWE_BENCH — 500 queries



✅ 15,144 queries extracted


════════════════════════════════════════════════════════════
  GLOBAL — all datasets pooled
════════════════════════════════════════════════════════════
                mean      std     min        max    p1      p5      p10      p25      p50      p75       p90       p95    p99
feature                                                                                                                      
token_count  48.1797  71.9024  1.0000  2417.0000  5.00  8.0000  10.0000  15.0000  26.0000  52.0000  108.0000  167.0000  324.0
ner_density   0.1029   0.0963  0.0000     0.8333  0.00  0.0000   0.0000   0.0182   0.0858   0.1562    0.2308    0.2899    0.4
mattr         0.8724   0.0877  0.3338     1.0000  0.65  0.7265   0.7633   0.8162   0.8720   0.9333    1.0000    1.0000    1.0

════════════════════════════════════════════════════════════
  AIME — 30 queries
════════════════════════════════════════════════════════════
                mean      std      min      

## L2: Reasoning Depth (R)

**Captures:** cognitive demand, structural complexity, and discourse-level reasoning.

**Sub-features:**

| Sub-feature | Signal | Proxy |
|-------------|--------|-------|
| `bloom_level` | Cognitive verb tier (1–6) | Regex over Bloom's action verbs |
| `syntactic_depth` | Max dependency tree depth | spaCy parser subtree recursion |
| `multi_hop` | Cross-entity reasoning steps | Boolean: 2+ distinct entity chains |
| `negation_count` | Logical inversions | `neg` dependency arcs |
| `conditional_count` | Hypothetical/causal reasoning | Conditional conjunctions |
| `modifier_density` | Adjectival/adverbial qualifiers | amod + advmod arc density |

In [9]:
# ── Bloom's Taxonomy verb lexicon (levels 1–6, compiled once) ───────────────
_BLOOM_VERBS: Dict[int, List[str]] = {
    1: [  # Remember — surface recall
        "define","list","recall","name","identify","recognize","state","label",
        "match","select","locate","arrange","duplicate","memorize","repeat",
        "reproduce","copy","quote","order","record","relate","underline",
        "who is","when was","where is","what year","what is the capital",
    ],
    2: [  # Understand — interpretation, paraphrase
        "explain","describe","interpret","paraphrase","restate","translate",
        "clarify","elaborate","simplify","convert","classify","categorize",
        "sort","group","infer","predict","conclude from","anticipate",
        "extrapolate","summarize","abstract","generalize","outline",
        "illustrate","give an example","represent","compare","report",
        "review","discuss","indicate","rewrite","what is the difference",
    ],
    3: [  # Apply — procedural, computational tasks
        "apply","use","implement","execute","carry out","perform","administer",
        "operate","employ","utilize","solve","compute","calculate","estimate",
        "determine","find","derive the value","work out","demonstrate",
        "show how to","illustrate how","construct","build","produce","make",
        "prepare","complete","modify","develop a solution","simulate","run",
        "test","sketch","map","write a function","code","script",
    ],
    4: [  # Analyze — decompose, compare, attribute
        "analyze","analyse","differentiate","distinguish","discriminate",
        "separate","decompose","deconstruct","break down","parse","dissect",
        "organize","structure","integrate","diagram","correlate",
        "attribute","deduce","detect","examine","investigate",
        "compare and contrast","contrast","examine differences",
        "compare the trade-offs","trade-off","trade-offs",
        "what are the trade-offs","explain why","what causes","trace",
        "map the relationship","identify assumptions","identify biases",
        "debate","prioritize",
    ],
    5: [  # Evaluate — judge, critique, defend
        "evaluate","assess","judge","critique","criticize","review critically",
        "appraise","rate","rank","check","monitor","validate","verify","debug",
        "judge whether","measure","inspect","score","justify","defend",
        "argue for","argue against","support","refute","counter","recommend",
        "decide","choose between","determine the best","conclude",
        "is it better to","should we","which is more","what is the optimal",
    ],
    6: [  # Create — synthesize, design, prove
        "design","plan","devise","propose","formulate","architect","blueprint",
        "create","develop","generate","produce","invent","originate","pioneer",
        "compose","author","draft","synthesize","combine","assemble","compile",
        "derive","prove","demonstrate that","show that","formally show",
        "from first principles","from scratch","hypothesize","theorize",
        "conjecture","postulate","novel","new approach","new method",
        "original","come up with","propose a new","imagine a new",
    ],
}

# Pre-compile sorted patterns: longest phrases first to avoid partial matches
_BLOOM_PATTERNS: Dict[int, re.Pattern] = {
    lvl: re.compile(
        r"\b(?:" + "|".join(
            re.escape(v) for v in sorted(verbs, key=len, reverse=True)
        ) + r")\b",
        re.IGNORECASE,
    )
    for lvl, verbs in _BLOOM_VERBS.items()
}

# Conditional conjunctions indicating hypothetical / causal reasoning
_CONDITIONAL_TOKENS = frozenset([
    "if","unless","provided","assuming","given","suppose","suppose that",
    "in the event","whenever","only if","even if","as long as",
    "because","since","therefore","thus","hence","consequently",
    "as a result","due to","owing to","so that","in order to",
])


class ReasoningDepth:
    """
    L2 — Reasoning Depth.

    Measures *cognitive* and *structural* complexity via:
      - Bloom's taxonomy level (1–6)  →  what cognitive operation is demanded
      - Syntactic tree depth          →  structural embedding of clauses
      - Multi-hop indicator           →  does the query chain across entities/steps
      - Negation count                →  logical inversions
      - Conditional count             →  hypothetical / causal constructs
      - Modifier density              →  adjectival + adverbial qualifiers

    All sub-features are normalised and combined into R ∈ [0, 1].
    """

    # Sub-feature weights (must sum to 1.0)
    _W = {
        "bloom":     0.30,
        "syn_depth": 0.20,
        "multi_hop": 0.15,
        "negation":  0.10,
        "condition": 0.15,
        "modifier":  0.10,
    }

    def __init__(self, text: str, doc: Optional[spacy.tokens.Doc] = None):
        self.text = text
        self.doc  = doc if doc is not None else get_doc(text)

    # ── Sub-features ─────────────────────────────────────────────────────────

    def _bloom_level(self) -> int:
        """
        Return the *highest* Bloom level matched in the text (1–6, default 1).
        Longer phrases are matched first (pre-sorted in _BLOOM_PATTERNS).
        """
        best = 1
        for lvl in range(6, 0, -1):   # check highest level first
            if _BLOOM_PATTERNS[lvl].search(self.text):
                best = lvl
                break
        return best

    def _syntactic_depth(self) -> float:
        """
        Mean maximum dependency-tree depth over all sentences.
        Higher = more deeply embedded clause structure.
        """
        depths = []
        for sent in self.doc.sents:
            root = sent.root
            depths.append(_tree_depth(root))
        return round(float(np.mean(depths)) if depths else 0.0, 4)

    def _is_multi_hop(self) -> bool:
        """
        Heuristic: query is multi-hop if it contains ≥2 distinct named entities
        AND at least one relational / comparison token.
        """
        entity_types = {ent.label_ for ent in self.doc.ents}
        if len(entity_types) < 2:
            return False
        relational_tokens = {"and","between","both","versus","vs","compared",
                             "relate","link","connect","differ","same","than"}
        query_tokens = {t.lower_ for t in self.doc}
        return bool(query_tokens & relational_tokens)

    def _negation_count(self) -> int:
        """Count tokens with a `neg` dependency arc."""
        return sum(1 for t in self.doc if t.dep_ == "neg")

    def _conditional_count(self) -> int:
        """Count conditional/causal conjunctions using the pre-defined lexicon."""
        tokens_lower = {t.lower_ for t in self.doc}
        return len(tokens_lower & _CONDITIONAL_TOKENS)

    def _modifier_density(self) -> float:
        """
        Ratio of adjectival (amod) + adverbial (advmod) modifiers to total tokens.
        Denser modifiers → more constrained, nuanced query.
        """
        total = max(len(self.doc), 1)
        mods  = sum(1 for t in self.doc if t.dep_ in {"amod", "advmod"})
        return round(mods / total, 4)

    # ── Public API ───────────────────────────────────────────────────────────

    def raw_features(self) -> dict:
        return {
            "bloom_level":      self._bloom_level(),
            "syntactic_depth":  self._syntactic_depth(),
            "multi_hop":        int(self._is_multi_hop()),
            "negation_count":   self._negation_count(),
            "conditional_count":self._conditional_count(),
            "modifier_density": self._modifier_density(),
        }

    def score(self) -> Tuple[float, dict]:
        """
        Returns (R, raw_features).
        R is the normalised reasoning complexity ∈ [0, 1].
        """
        raw = self.raw_features()

        # Bloom: levels 1–6 → scale to [0,1]
        s_bloom    = (raw["bloom_level"] - 1) / 5.0
        # Syntactic depth: typical range [2, 15]
        s_depth    = min_max_scale(raw["syntactic_depth"], 1.0, 15.0)
        # Multi-hop: binary
        s_hop      = float(raw["multi_hop"])
        # Negation: 0–4 typical range
        s_neg      = min_max_scale(raw["negation_count"], 0, 4)
        # Conditionals: 0–5 typical range
        s_cond     = min_max_scale(raw["conditional_count"], 0, 5)
        # Modifier density: 0–0.25 typical range
        s_mod      = min_max_scale(raw["modifier_density"], 0.0, 0.5)

        W = self._W
        R = (W["bloom"]     * s_bloom  +
             W["syn_depth"] * s_depth  +
             W["multi_hop"] * s_hop    +
             W["negation"]  * s_neg    +
             W["condition"] * s_cond   +
             W["modifier"]  * s_mod)
        return fix(R), raw


# ── Sanity check ────────────────────────────────────────────────────────────
_demo_r = "Analyze and compare the trade-offs between transformer and LSTM architectures for long-range dependency modelling."
_rd     = ReasoningDepth(sample)
_R, _rr = _rd.score()
print(f"Demo : '{sample[:70]}...'")
print(f"Raw  : {_rr}")
print(f"R    : {_R:.4f}")

Demo : 'A paper about AI regulation that was originally submitted to arXiv.org...'
Raw  : {'bloom_level': 2, 'syntactic_depth': 8.0, 'multi_hop': 1, 'negation_count': 0, 'conditional_count': 0, 'modifier_density': 0.0339}
R    : 0.3168


## Cell 5 — L3: Tool Dependency (T)

**Captures:** whether the task requires external tool calls, APIs, code execution, or system interaction — a strong proxy for agentic complexity.  

**Method:** Keyword / regex scan over a curated tool-signal lexicon, grouped into categories.  
T ∈ [0, 1] is proportional to the number of distinct tool categories triggered.

In [10]:
# ── Tool-signal lexicon grouped by category ──────────────────────────────────
_TOOL_SIGNALS: Dict[str, List[str]] = {
    "code_execution":  [
        "run","execute","compile","script","program","code","function",
        "algorithm","implement","debug","unit test","pytest","bash","terminal",
        "command line","cli","shell","subprocess","notebook","jupyter",
    ],
    "web_search":      [
        "search","browse","look up","find online","google","bing","retrieve",
        "latest","current","real-time","live","up-to-date","news",
        "fetch","scrape","crawl","url","website","webpage",".com",".org",".in",
    ],
    "file_io":         [
        "read file","write file","open file","save to","load from","csv","json",
        "excel","pdf","word doc","docx","txt","download","upload","import",
        "export","parse document","extract from","dataset",
    ],
    "api_call":        [
        "api","rest","graphql","endpoint","request","response","http","https",
        "curl","webhook","oauth","token","authenticate","authorize",
        "call service","microservice","grpc",
    ],
    "database":        [
        "database","sql","query","select","insert","update","delete","join",
        "table","schema","mongodb","postgres","mysql","sqlite","redis",
        "elasticsearch","nosql",
    ],
    "external_service": [
        "slack","email","calendar","send message","notify","alert",
        "github","jira","trello","confluence","zapier","ifttt",
        "gpt","llm","openai","anthropic","claude","gemini",
    ],
    "math_computation": [
        "calculate","compute","integral","derivative","matrix","solve equation",
        "eigenvalue","optimize","gradient","probability","statistics",
        "regression","fourier","laplace",
    ],
    "multimodal":      [
        "image","photo","picture","diagram","chart","graph","plot","video",
        "audio","speech","transcribe","ocr","caption","visualize",
    ],
}

# Compile each category into a single regex
_TOOL_PATTERNS: Dict[str, re.Pattern] = {
    cat: re.compile(
        r"\b(?:" + "|".join(re.escape(t) for t in sorted(terms, key=len, reverse=True)) + r")\b",
        re.IGNORECASE,
    )
    for cat, terms in _TOOL_SIGNALS.items()
}


class ToolDependency:
    """
    L3 — Tool Dependency.

    Identifies which external tool *categories* the task likely requires and
    returns a normalised score T ∈ [0, 1] based on the count of distinct
    categories triggered.

    Having 1 tool category → moderate complexity;  
    Having 4+ categories   → high agentic complexity (T → 1.0).
    """

    _MAX_CATS = 4   # saturation point

    def __init__(self, text: str):
        self.text = text

    def raw_features(self) -> dict:
        hits = {cat: bool(_TOOL_PATTERNS[cat].search(self.text))
                for cat in _TOOL_PATTERNS}
        return {
            "matched_categories": [c for c, v in hits.items() if v],
            "category_count":     sum(hits.values()),
            "category_flags":     hits,
        }

    def score(self) -> Tuple[float, dict]:
        """
        Returns (T, raw_features).  T ∈ [0, 1].
        T = min(category_count / MAX_CATS, 1.0)
        """
        raw = self.raw_features()
        T   = fix(raw["category_count"] / self._MAX_CATS)
        return T, raw


## L4: Domain Skills (D)

**Captures:** breadth and depth of domain knowledge required.  

**Sub-features:**
- `domain_count` → how many distinct knowledge domains are referenced  
- `temporal_complexity` → presence of date arithmetic, historical context, or future prediction

D ∈ [0, 1] grows with multi-domain queries and temporal reasoning demands.

In [11]:
_DOMAIN_SIGNALS: Dict[str, List[str]] = {
    "mathematics":   [
        "theorem","proof","lemma","calculus","algebra","geometry","topology",
        "number theory","combinatorics","probability","statistics","integral",
        "derivative","polynomial","matrix","vector space","differential equation",
    ],
    "computer_science": [
        "algorithm","data structure","complexity","big-o","recursion","sorting",
        "graph","tree","dynamic programming","machine learning","neural network",
        "transformer","attention","gradient","backpropagation","compiler",
        "operating system","concurrency","parallelism","distributed","blockchain",
    ],
    "science":       [
        "physics","chemistry","biology","quantum","thermodynamics","genetics",
        "protein","molecule","atom","electron","cell","dna","rna","enzyme",
        "evolution","ecology","astronomy","relativity","nuclear",
    ],
    "medicine":      [
        "diagnosis","symptom","treatment","drug","clinical","patient",
        "surgery","pathology","pharmacology","dosage","side effect",
        "trial","prognosis","cancer","diabetes","blood pressure",
    ],
    "law":           [
        "statute","regulation","legal","law","court","jurisdiction",
        "precedent","contract","liability","intellectual property",
        "gdpr","compliance","patent","copyright","tort",
    ],
    "finance":       [
        "stock","bond","option","derivative","portfolio","hedge","equity",
        "valuation","balance sheet","income statement","cash flow","roi",
        "interest rate","inflation","gdp","monetary policy",
    ],
    "history_culture":[
        "war","empire","revolution","civilization","ancient","medieval",
        "renaissance","colonialism","culture","philosophy","religion",
        "mythology","archaeology","linguistics",
    ],
    "engineering":   [
        "circuit","electrical","mechanical","structural","thermal","fluid",
        "signal processing","control system","robotics","embedded",
        "fpga","microcontroller","antenna","semiconductor",
    ],
}

_DOMAIN_PATTERNS: Dict[str, re.Pattern] = {
    dom: re.compile(
        r"\b(?:" + "|".join(re.escape(k) for k in sorted(kw, key=len, reverse=True)) + r")\b",
        re.IGNORECASE,
    )
    for dom, kw in _DOMAIN_SIGNALS.items()
}

# Temporal complexity indicators
_TEMPORAL_SIGNALS = [
    re.compile(r"\b(\d{4}|\d{1,2}/\d{1,2}/\d{2,4})\b"),          # explicit dates
    re.compile(r"\b(before|after|since|until|during|between|by|\d+\s*(years?|months?|days?|decades?))\b", re.I),
    re.compile(r"\b(historical|future|predict|forecast|trend|evolve|timeline)\b", re.I),
    re.compile(r"\b(current|latest|recent|today|now|as of)\b", re.I),
]


class DomainSkills:
    """
    L4 — Domain Skills.

    Measures cross-domain breadth and temporal reasoning:
      - domain_count         : number of distinct knowledge domains hit
      - temporal_complexity  : number of distinct temporal signal patterns matched

    D ∈ [0, 1] saturates at 4+ domains or 3+ temporal signals.
    """

    _W_DOMAIN   = 0.65
    _W_TEMPORAL = 0.35
    _MAX_DOMAIN   = 4    # saturation point for domain count
    _MAX_TEMPORAL = 3    # saturation point for temporal signals

    def __init__(self, text: str):
        self.text = text

    def raw_features(self) -> dict:
        domains_hit  = {d for d, pat in _DOMAIN_PATTERNS.items() if pat.search(self.text)}
        temp_signals = sum(1 for pat in _TEMPORAL_SIGNALS if pat.search(self.text))
        return {
            "matched_domains":   sorted(domains_hit),
            "domain_count":      len(domains_hit),
            "temporal_signals":  temp_signals,
        }

    def score(self) -> Tuple[float, dict]:
        """Returns (D, raw_features).  D ∈ [0, 1]."""
        raw = self.raw_features()
        s_domain   = min_max_scale(raw["domain_count"],     0, self._MAX_DOMAIN)
        s_temporal = min_max_scale(raw["temporal_signals"], 0, self._MAX_TEMPORAL)
        D = self._W_DOMAIN * s_domain + self._W_TEMPORAL * s_temporal
        return fix(D), raw


# ── Sanity check ─────────────────────────────────────────────────────────────
_demo_d = "Compare the historical evolution of quantum computing hardware since 2000 and predict the next decade's trajectory."
_ds     = DomainSkills(sample)
_D, _rd2 = _ds.score()
print(f"Demo    : '{sample[:70]}...'")
print(f"Domains : {_rd2['matched_domains']}  (count={_rd2['domain_count']})")
print(f"Temporal: {_rd2['temporal_signals']} signal(s)")
print(f"D       : {_D:.4f}")

Demo    : 'A paper about AI regulation that was originally submitted to arXiv.org...'
Domains : ['law', 'science']  (count=2)
Temporal: 1 signal(s)
D       : 0.4417


## Cell 7 — L5: Task Type (TT)

**Captures:** the structural *output* demanded by the task.  
Complexity inherent in the expected response format:

| Type | Score | Examples |
|------|-------|----------|
| Generative | 1.0 | Compose, write, design, create, generate |
| Reasoning | 0.8 | Prove, analyze, explain why, derive |
| Alt-choice | 0.5 | Which is better, compare options, select best |
| Single-answer | 0.3 | What year, who is, where is |
| Boolean/Decision | 0.15 | Is X true, yes/no, does X exist |

TT is the highest-scoring type matched in the query.

In [12]:
# Task type definitions: (name, score, trigger_patterns)
_TASK_TYPE_RULES: List[Tuple[str, float, re.Pattern]] = [
    (
        "generative", 1.0,
        re.compile(
            r"\b(write|compose|draft|generate|create|design|build|develop|produce"
            r"|author|invent|synthesize|formulate|devise|architect|plan a|propose a"
            r"|construct|make a|come up with)\b",
            re.I,
        ),
    ),
    (
        "reasoning", 0.8,
        re.compile(
            r"\b(prove|derive|show that|explain why|analyze|analyse|justify"
            r"|evaluate|assess|critique|reason|investigate|demonstrate that"
            r"|how does|why does|what causes|what leads to|infer|conclude)\b",
            re.I,
        ),
    ),
    (
        "alt_choice", 0.5,
        re.compile(
            r"\b(which is better|compare|choose between|select the best|best option"
            r"|trade-off|versus|vs\.?|alternatives|should i use|recommend|rank"
            r"|what would you prefer|which approach)\b",
            re.I,
        ),
    ),
    (
        "single_answer", 0.3,
        re.compile(
            r"\b(what is|who is|where is|when was|what year|how many|how much"
            r"|what was|name the|list the|define|what does .* mean|capital of)\b",
            re.I,
        ),
    ),
    (
        "boolean_decision", 0.15,
        re.compile(
            r"\b(is it|is there|are there|does .* exist|can .* be|is .* true"
            r"|do .* have|has .* ever|was .* ever|would .* work|is .* possible)\b",
            re.I,
        ),
    ),
]


class TaskType:
    """
    L5 — Task Type.

    Classifies the *structural output demanded* by the query and maps it to a
    complexity score TT ∈ [0, 1].  Evaluation follows a priority cascade:
    the first (highest-complexity) matching rule wins.

    Types, in descending order of complexity:
      generative (1.0) > reasoning (0.8) > alt_choice (0.5)
      > single_answer (0.3) > boolean_decision (0.15)
    """

    def __init__(self, text: str):
        self.text = text

    def raw_features(self) -> dict:
        matched = []
        for name, score, pattern in _TASK_TYPE_RULES:
            if pattern.search(self.text):
                matched.append({"type": name, "score": score})
        # Highest complexity type is the first in the sorted list
        best = matched[0] if matched else {"type": "unknown", "score": 0.2}
        return {
            "primary_type":  best["type"],
            "primary_score": best["score"],
            "all_matched":   matched,
        }

    def score(self) -> Tuple[float, dict]:
        """Returns (TT, raw_features).  TT ∈ [0, 1]."""
        raw = self.raw_features()
        return fix(raw["primary_score"]), raw


# ── Sanity check ─────────────────────────────────────────────────────────────
for _q in [
    "Write a novel short story about AI consciousness.",
    "Analyze why LSTM vanishing gradient problem occurs.",
    "Which is better: PyTorch or TensorFlow for research?",
    "What year was Python first released?",
    "Is transformer architecture Turing complete?",
]:
    _tt = TaskType(_q)
    _TT, _rtt = _tt.score()
    print(f"[{_rtt['primary_type']:20s} | TT={_TT:.2f}]  '{_q[:55]}'")

[generative           | TT=1.00]  'Write a novel short story about AI consciousness.'
[reasoning            | TT=0.80]  'Analyze why LSTM vanishing gradient problem occurs.'
[alt_choice           | TT=0.50]  'Which is better: PyTorch or TensorFlow for research?'
[single_answer        | TT=0.30]  'What year was Python first released?'
[unknown              | TT=0.20]  'Is transformer architecture Turing complete?'


## Cell 8 — Composite Scorer: C(q)

Combines all 5 layers into the final complexity score:

$$C(q) = w_S \cdot S + w_R \cdot R + w_T \cdot T + w_D \cdot D + w_{TT} \cdot TT$$

Default weights are grounded in the professor's framework (Reasoning is the dominant signal).

In [ ]:
@dataclass
class ComplexityResult:
    """Full result container for one query."""
    query:          str
    C:              float           # final composite score ∈ [0,1]
    S:              float
    R:              float
    T:              float
    D:              float
    TT:             float
    raw_S:          dict = field(repr=False)
    raw_R:          dict = field(repr=False)
    raw_T:          dict = field(repr=False)
    raw_D:          dict = field(repr=False)
    raw_TT:         dict = field(repr=False)
    complexity_band: str = ""       # human-readable tier

    def __post_init__(self):
        if   self.C < 0.20:  self.complexity_band = "🟢 LOW"
        elif self.C < 0.45:  self.complexity_band = "🟡 MEDIUM"
        elif self.C < 0.60:  self.complexity_band = "🟠 HIGH"
        else:                self.complexity_band = "🔴 VERY HIGH"

    def summary(self) -> str:
        return (
            f"C={self.C:.3f} {self.complexity_band}  "
            f"[S={self.S:.2f} R={self.R:.2f} T={self.T:.2f} "
            f"D={self.D:.2f} TT={self.TT:.2f}]"
        )


class TaskComplexityEstimator:
    """
    Model-Agnostic Task Complexity Estimator.

    Computes  C(q) = w_S*S + w_R*R + w_T*T + w_D*D + w_TT*TT
    entirely from classical NLP features — zero LLM calls.

    Parameters
    ----------
    weights : dict, optional
        Custom weights for each dimension.  Must include keys
        'S', 'R', 'T', 'D', 'TT'.  Automatically renormalized to sum=1.
    """

    # Default weights — calibrated so reasoning is the dominant signal,
    # followed by task type, then surface / tool / domain.
    _DEFAULT_WEIGHTS = {
        "S":  0.15,   # Surface Features
        "R":  0.35,   # Reasoning Depth  
        "T":  0.20,   # Tool Dependency
        "D":  0.15,   # Domain Skills
        "TT": 0.15,   # Task Type
    }

    def __init__(self, weights: Optional[Dict[str, float]] = None):
        raw_w = weights or self._DEFAULT_WEIGHTS
        total = sum(raw_w.values())
        self.weights = {k: v / total for k, v in raw_w.items()}  # normalise

    def score(self, query: str) -> ComplexityResult:
        """Estimate the complexity of a single query string."""
        # Parse once, share the Doc across L1 and L2
        doc = get_doc(query)

        S,  raw_S  = SurfaceFeatures(query, doc=doc).score()
        R,  raw_R  = ReasoningDepth(query,  doc=doc).score()
        T,  raw_T  = ToolDependency(query).score()
        D,  raw_D  = DomainSkills(query).score()
        TT, raw_TT = TaskType(query).score()

        w = self.weights
        C = fix(w["S"]*S + w["R"]*R + w["T"]*T + w["D"]*D + w["TT"]*TT)

        return ComplexityResult(
            query=query, C=C,
            S=S, R=R, T=T, D=D, TT=TT,
            raw_S=raw_S, raw_R=raw_R, raw_T=raw_T,
            raw_D=raw_D, raw_TT=raw_TT,
        )

    def score_batch(self, queries: List[str]) -> pd.DataFrame:
        """Score a list of queries and return a tidy DataFrame."""
        records = []
        for q in queries:
            r = self.score(q)
            records.append({
                "query":            q[:80] + ("..." if len(q) > 80 else ""),
                "C":                round(r.C,  3),
                "S":                round(r.S,  3),
                "R":                round(r.R,  3),
                "T":                round(r.T,  3),
                "D":                round(r.D,  3),
                "TT":               round(r.TT, 3),
                "band":             r.complexity_band,
                "task_type":        r.raw_TT["primary_type"],
                "bloom_level":      r.raw_R["bloom_level"],
                "tool_categories":  ", ".join(r.raw_T["matched_categories"]) or "none",
                "domains":          ", ".join(r.raw_D["matched_domains"]) or "none",
            })
        return pd.DataFrame(records)


# ── Instantiate global estimator ─────────────────────────────────────────────
estimator = TaskComplexityEstimator()
print(f"✅ Estimator ready.  Effective weights: {estimator.weights}")

✅ Estimator ready.  Effective weights: {'S': 0.15, 'R': 0.35, 'T': 0.2, 'D': 0.15, 'TT': 0.15}


## Cell 9 — Benchmark Task Suite

We test on a diverse set of tasks spanning different benchmarks mentioned in the professor's discussion:
- **Simple factoid** (GSM8K-easy, MMLU recall)
- **Math reasoning** (GSM8K-hard, MATH competition)
- **Code / SWE** (SWE-bench style)
- **Multi-hop QA** (HotpotQA, MuSiQue)
- **Agentic / Tool-use** (GAIA, ToolBench)
- **Scientific reasoning** (GPQA)

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Score Task Complexity for All Datasets
# ─────────────────────────────────────────────────────────────────────────────

from scipy.stats import spearmanr
import pandas as pd
import numpy as np

QUERY_COL_MAP: Dict[str, str] = {
    "gaia"      : "query",
    "aime"      : "query",
    "mmlu_pro"  : "query",
    "musique"   : "query",
    "swe_bench" : "query",
}

# Known difficulty ordering for cross-dataset Spearman
# Higher rank = harder (based on established benchmark difficulty consensus)
EXPECTED_DIFFICULTY_RANK: Dict[str, int] = {
    "musique"   : 1,   # easiest  — structured multi-hop QA
    "mmlu_pro"  : 2,   # multiple choice, broad but bounded
    "aime"      : 3,   # competition math
    "gaia"      : 4,   # real-world multi-step tasks
    "swe_bench" : 5,   # hardest  — real software engineering
}

SUB_DIMS = ["S", "R", "T", "D", "TT"]

scored_datasets: Dict[str, pd.DataFrame] = {}

for name, df in datasets.items():
    query_col = QUERY_COL_MAP.get(name, "query")

    # ── Validate the query column exists ────────────────────────────────────
    if query_col not in df.columns:
        available = [
            c for c in df.columns
            if "query" in c.lower() or "question" in c.lower()
        ]
        raise KeyError(
            f"[{name}] Expected column '{query_col}' not found. "
            f"Possible candidates: {available or list(df.columns)}"
        )

    queries: List[str] = (
        df[query_col]
        .fillna("")
        .astype(str)
        .tolist()
    )

    print(f"\n{'─' * 60}")
    print(f"⚙️  Scoring  : {name.upper()}  ({len(queries):,} queries)")

    # ── Batch score ──────────────────────────────────────────────────────────
    scored_df: pd.DataFrame = estimator.score_batch(queries)

    # ── Merge complexity columns back onto the original DataFrame ────────────
    complexity_cols = [
        "C", "S", "R", "T", "D", "TT",
        "band", "task_type", "bloom_level",
        "tool_categories", "domains",
    ]

    merged_df = pd.concat(
        [df.reset_index(drop=True), scored_df[complexity_cols].reset_index(drop=True)],
        axis=1,
    )

    scored_datasets[name] = merged_df

    # ── Per-dataset summary ──────────────────────────────────────────────────
    # FIX: use clean band labels without emoji for reliable string matching
    band_clean = merged_df["band"].str.extract(r"(LOW|MEDIUM|HIGH|VERY HIGH)")[0]

    band_counts = merged_df["band"].value_counts()
    n = len(merged_df)

    print(f"   Rows scored : {n:,}")
    print(f"   Avg C score : {merged_df['C'].mean():.3f}  "
          f"(min={merged_df['C'].min():.3f}, max={merged_df['C'].max():.3f})")
    print(f"   Band distribution:")
    for band, count in band_counts.items():
        pct = count / n * 100
        print(f"     {band:<24} {count:>6,}  ({pct:5.1f}%)")

    # ── Save individual file ─────────────────────────────────────────────────
    out_path = OUTPUT_DIR / f"{name}_complexity.parquet"
    merged_df.to_parquet(out_path, index=False)
    print(f"\n   💾 Saved → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Cross-Dataset Summary Table + Cross-Dataset Spearman ρ
# ─────────────────────────────────────────────────────────────────────────────

summary_rows = []
for name, df in scored_datasets.items():
    # FIX: extract clean band label before percentage calculations
    band_clean = df["band"].str.extract(r"(LOW|MEDIUM|HIGH|VERY HIGH)")[0]

    summary_rows.append({
        "dataset"    : name,
        "n_queries"  : len(df),
        "mean_C"     : round(df["C"].mean(),  3),
        "mean_S"     : round(df["S"].mean(),  3),
        "mean_R"     : round(df["R"].mean(),  3),
        "mean_T"     : round(df["T"].mean(),  3),
        "mean_D"     : round(df["D"].mean(),  3),
        "mean_TT"    : round(df["TT"].mean(), 3),
        # FIX: use clean band series — no emoji interference
        "pct_LOW"    : round((band_clean == "LOW").sum()       / len(df) * 100, 1),
        "pct_MEDIUM" : round((band_clean == "MEDIUM").sum()    / len(df) * 100, 1),
        "pct_HIGH"   : round((band_clean == "HIGH").sum()      / len(df) * 100, 1),
        "pct_VHIGH"  : round((band_clean == "VERY HIGH").sum() / len(df) * 100, 1),
    })

summary_df = pd.DataFrame(summary_rows).set_index("dataset")

# ── Cross-dataset Spearman ρ: mean_C vs expected difficulty rank ─────────────
summary_df["expected_rank"] = summary_df.index.map(EXPECTED_DIFFICULTY_RANK)

rho_cross, pval_cross = spearmanr(
    summary_df["mean_C"],
    summary_df["expected_rank"],
)

# ── Sub-dimension cross-dataset Spearman ────────────────────────────────────
print("\n" + "═" * 60)
print("📊 CROSS-DATASET COMPLEXITY SUMMARY")
print("═" * 60)
print(summary_df.drop(columns="expected_rank").to_string())

# ── Save outputs ─────────────────────────────────────────────────────────────
summary_path = OUTPUT_DIR / "all_datasets_complexity_summary.parquet"
summary_df.to_parquet(summary_path)

print(f"\n✅ All files saved to  → {OUTPUT_DIR}/")
print(f"   Summary table saved → {summary_path}")


────────────────────────────────────────────────────────────
⚙️  Scoring  : GAIA  (165 queries)


   Rows scored : 165
   Avg C score : 0.257  (min=0.124, max=0.507)
   Band distribution:
     🟡 MEDIUM                    133  ( 80.6%)
     🟢 LOW                        26  ( 15.8%)
     🟠 HIGH                        6  (  3.6%)

   💾 Saved → data/processed/gaia_complexity.parquet

────────────────────────────────────────────────────────────
⚙️  Scoring  : AIME  (30 queries)
   Rows scored : 30
   Avg C score : 0.237  (min=0.166, max=0.355)
   Band distribution:
     🟡 MEDIUM                     27  ( 90.0%)
     🟢 LOW                         3  ( 10.0%)

   💾 Saved → data/processed/aime_complexity.parquet

────────────────────────────────────────────────────────────
⚙️  Scoring  : MMLU_PRO  (12,032 queries)
   Rows scored : 12,032
   Avg C score : 0.200  (min=0.040, max=0.626)
   Band distribution:
     🟢 LOW                     5,958  ( 49.5%)
     🟡 MEDIUM                  5,905  ( 49.1%)
     🟠 HIGH                      166  (  1.4%)
     🔴 VERY HIGH                   3  (  0.0%)

In [26]:
data_summary = pd.read_parquet("data/processed/all_datasets_complexity_summary.parquet")

In [27]:
data_summary

,n_queries,mean_C,mean_S,mean_R,mean_T,mean_D,mean_TT,pct_LOW,pct_MEDIUM,pct_HIGH,pct_VHIGH,expected_rank
dataset,,,,,,,,,,,,
gaia,165,0.257,0.516,0.246,0.129,0.144,0.308,15.8,80.6,3.6,0.0,4
aime,30,0.237,0.504,0.323,0.033,0.072,0.207,10.0,90.0,0.0,0.0,3
mmlu_pro,12032,0.200,0.427,0.207,0.053,0.091,0.264,49.5,49.1,1.4,0.0,2
musique,2417,0.163,0.407,0.150,0.007,0.047,0.272,74.3,25.7,0.0,0.0,1
swe_bench,500,0.433,0.556,0.396,0.607,0.192,0.401,4.4,50.2,30.8,14.6,5


In [28]:
data_all = pd.read_parquet("data/processed/swe_bench_complexity.parquet")

In [29]:
data_all[["difficulty", "band"]]

,difficulty,band
0,15 min - 1 hour,🟡 MEDIUM
1,15 min - 1 hour,🟠 HIGH
2,15 min - 1 hour,🔴 VERY HIGH
3,1-4 hours,🔴 VERY HIGH
4,15 min - 1 hour,🔴 VERY HIGH
...,...,...
495,15 min - 1 hour,🟡 MEDIUM
496,15 min - 1 hour,🟡 MEDIUM
497,<15 min fix,🟡 MEDIUM
498,<15 min fix,🟡 MEDIUM


In [30]:
data_all["difficulty"].value_counts()

difficulty
15 min - 1 hour    261
<15 min fix        194
1-4 hours           42
>4 hours             3
Name: count, dtype: int64

In [31]:
data_all["band"].value_counts()

band
🟡 MEDIUM       251
🟠 HIGH         154
🔴 VERY HIGH     73
🟢 LOW           22
Name: count, dtype: int64

In [32]:
mapping = {
    "<15 min fix": "🟢 LOW",
    "15 min - 1 hour": "🟡 MEDIUM",
    "1-4 hours": "🟠 HIGH",
    ">4 hours": "🔴 VERY HIGH",
}

data_all["band_from_difficulty"] = data_all["difficulty"].map(mapping)

In [33]:
(data_all["band"] == data_all["band_from_difficulty"]).value_counts()


False    348
True     152
Name: count, dtype: int64

In [34]:
data_all[data_all["band"] != data_all["band_from_difficulty"]][
    ["difficulty", "band", "band_from_difficulty"]
]

,difficulty,band,band_from_difficulty
1,15 min - 1 hour,🟠 HIGH,🟡 MEDIUM
2,15 min - 1 hour,🔴 VERY HIGH,🟡 MEDIUM
3,1-4 hours,🔴 VERY HIGH,🟠 HIGH
4,15 min - 1 hour,🔴 VERY HIGH,🟡 MEDIUM
6,15 min - 1 hour,🟠 HIGH,🟡 MEDIUM
...,...,...,...
489,15 min - 1 hour,🟠 HIGH,🟡 MEDIUM
491,<15 min fix,🟠 HIGH,🟢 LOW
492,15 min - 1 hour,🔴 VERY HIGH,🟡 MEDIUM
497,<15 min fix,🟡 MEDIUM,🟢 LOW


(69.6, 30.4)

In [36]:
0.60


0.6